# 04 · Visualization & EDA

Pairs with `docs/01-data-pipeline.md` (step 5). Goal: see the data before trusting
any model. Uses the `gpulab.learn.viz` helpers and plain matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gpulab.learn import viz

## 1. A single curve

In [ ]:
# Reuse the synthetic generator idea so this runs without S3 data.
from scipy.signal import savgol_filter
def synth(n=24, nf=500, rng=None):
    rng = rng or np.random.default_rng(1); t = np.arange(nf); out=[]
    for _ in range(n):
        tm = rng.integers(390, 450)
        out.append(1/(1+np.exp((t-tm)/6.0)) + rng.normal(0,0.002,nf))
    return np.asarray(out)
from gpulab.data.preprocess import preprocess_curves, PreprocessConfig
X = preprocess_curves(synth(), PreprocessConfig())
viz.plot_curve(X[0], title="one preprocessed -dF/dT curve")
plt.show()

## 2. Many curves, colored by (toy) label

In [ ]:
y = np.random.default_rng(2).integers(0, 3, size=len(X))
viz.plot_curves(X, y, title="curves by class")
plt.show()

## 3. A batch as a heatmap

In [ ]:
viz.heatmap(X[:20], title="20 curves (rows) x 120 frames")
plt.show()

## 4. On your real data (after building the dataset)

Once `data/processed/dataset.npz` exists, swap the synthetic block for the real one
and look for structure: do curves of the same species cluster in peak position/width?

In [ ]:
from pathlib import Path
from gpulab.data.dataset import Dataset
p = Path("../data/processed/dataset.npz")
if p.exists():
    ds = Dataset.load(p)
    y, classes = ds.y_int()
    print(len(ds), "curves |", len(classes), "classes")
    viz.plot_curves(ds.X[:300], y[:300], title="real preprocessed curves")
    plt.show()
else:
    print("No dataset yet - build it first (docs/01-data-pipeline.md).")

> **Concepts to note** (copy into your own theory notebook):
> - Always look at the data before modeling; EDA catches framing/label bugs early.
> - Same-species curves should cluster by Tm (peak position) and width.
> - A heatmap of a batch is a fast way to spot outliers / all-identical curves.
> - `viz.*` returns a matplotlib Figure -> works in notebooks and scripts.